## 은닉층이 2개인 예제를 만들어보자

[문제] LogicGate클래스에서 첫번째 은닉층의 노드 갯수를 4개로 
두번째 은닉층의 노드 갯수를 2개로 변경하여 XOR게이트의 결과값이 정상적으로 나오도록 코드를 수정하여 제출하시오.

재출파일명 : 홍길동_0518.ipynb

<처리조건>
1. LogicGate클래스를 설계한다.
2. 학습후 정확도를 출력하도록 한다.

In [5]:
import numpy as np

def sigmoid(x):   # 시그모이드 함수
    return 1. / (1 + np.exp(-x))   # 수식을 표현한 내용

In [6]:
def numerical_derivative(f, x):   # f: 손실값을 계산하는 함수 / x: 업데이트 하기 위한 변수
    delta_x = 1e-4   # 미분할 값의 크기
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    
    while not it.finished:
        idx = it.multi_index        
        tmp_val = x[idx]
        x[idx] = float(tmp_val) + delta_x
        fx1 = f(x) # f(x+delta_x)
        
        x[idx] = float(tmp_val) - delta_x 
        fx2 = f(x) # f(x-delta_x)
        grad[idx] = (fx1 - fx2) / (2*delta_x)
        
        x[idx] = tmp_val 
        it.iternext()

    return grad

In [29]:
class LogicGate:
    def __init__(self, gate_name, xdata, tdata): # 생성자 : __init__ # 해당 클래스가 사용할 변수를 초기화할 목적으로 사용
        self.name = gate_name
        self.xdata = xdata.reshape(4,2)  # 입력층 데이터 // 입력층에 대한 설계
        self.tdata = tdata.reshape(4,1)  # 정답 데이터

        self.W2 = np.random.rand(2,4)    # 은닉층 가중치 // 은닉층에 대한 설계
        self.b2 = np.random.rand(4)      # 은닉층 바이어스

        self.W3 = np.random.rand(4,2)    # 은닉층 2
        self.b3 = np.random.rand(2)

        self.W4 = np.random.rand(2,1)    # 출력증 가중치 // 출력층에 대한 설계
        self.b4 = np.random.rand(1)      # 출력층 바이어스

        self.learning_rate = 1e-2        # 학습률

    def loss_val(self):                  # 손실값 계산 
        delta = 1e-7
        z2 = np.dot(self.xdata, self.W2) + self.b2 # 은닉층_1 회귀값 계산
        a2 = sigmoid(z2)                           # 은닉층_1 활성함수
        z3 = np.dot(a2, self.W3) + self.b3         # 은닉층_2 회귀값 계산
        a3 = sigmoid(z3)                           # 은닉층_2 활성함수
        z4 = np.dot(a3, self.W4) + self.b4         # 출력층 회귀값 계산
        y = a4 = sigmoid(z4)                       # 출력층 활성 함수
        # 크로스 엔트로피 계산
        return -np.sum(self.tdata*np.log(y+delta)+(1-self.tdata)*np.log((1 - y)+delta)) # 최종적인 y값을 가지고 손실값 계산

# 손실함수
    def train(self):
        f = lambda x : self.loss_val() # 손실함수 정의하는 형태 
        print("Initial loss value = ", self.loss_val()) # 초기손실값 계산하는 형태
        for step in range(500001): # 2만번 학습! >> 학습을 하게 되면 각 층에 있는 가중치와 바이어스를 업데이트 시켜야한다.
            self.W2 -= self.learning_rate * numerical_derivative(f, self.W2) # 은닉층의 가중치와 바이어스를 업데이트 시킴
            self.b2 -= self.learning_rate * numerical_derivative(f, self.b2) # 은닉층의 가중치와 바이어스를 업데이트 시킴
            self.W3 -= self.learning_rate * numerical_derivative(f, self.W3) # 은닉층_2의 가중치와 바이어스를 업데이트 시킴
            self.b3 -= self.learning_rate * numerical_derivative(f, self.b3) # 은닉층_2의 가중치와 바이어스를 업데이트 시킴
            self.W4 -= self.learning_rate * numerical_derivative(f, self.W4) # 출력층의 가중치와 바이어스를 업데이트 시킴
            self.b4 -= self.learning_rate * numerical_derivative(f, self.b4) # 출력층의 가중치와 바이어스를 업데이트 시킴
            if (step % 1000 == 0):                                           # 중간 확인 과정
                print("step = ", step, "loss value = ", self.loss_val())

    def predict(self, input_data):
        self.xdata = input_data
        z2 = np.dot(self.xdata, self.W2) + self.b2 # 은닉층에서 연산
        a2 = sigmoid(z2)                           # 은닉층에서 연산
        z3 = np.dot(a2, self.W3) + self.b3         # 은닉층_2
        a3 = sigmoid(z3)                           # 은닉층_2
        z4 = np.dot(self.xdata, self.W4) + self.b4 # 출력층에서 연산
        y = a4 = sigmoid(z4)                       # 출력층에서 연산
        if y > 0.5:
            result = 1
        else:
            result = 0

        return y, result

    # 정확도 계산하는 거 만들 거임
    def accuracy(self, test_xdata, test_tdata):
        matched_list = []                                               # 정답과 일치한 내용 저장
        not_matched_list = []                                           # 정답과 불일치한 내용 저장
        for index in range(len(test_xdata)):
            (real_val, logical_val) = self.predict(test_xdata[index])
            if logical_val == test_tdata[index]:
                matched_list.append(index)                              # 정답과 일치한 내용 추가
            else:
                not_matched_list.append(index)                          # 정답과 불일치한 내용 추가
        accuracy_val = len(matched_list) / len(test_xdata)              # 정확도 계산
        return accuracy_val

In [30]:
# XOR 게이트
# XOR
xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])   # 학습(train) 데이터
tdata = np.array([0, 1, 1, 0])                         # 정답(target) 데이터
XOR_obj = LogicGate("XOR_GATE", xdata, tdata)          # 모델 객체 생성
XOR_obj.train()                                        # 학습

Initial loss value =  3.450606660451671
step =  0 loss value =  3.421690201723717
step =  1000 loss value =  2.7721689543512054
step =  2000 loss value =  2.771995528131274
step =  3000 loss value =  2.7718123648578357
step =  4000 loss value =  2.7716162070301
step =  5000 loss value =  2.7714034565763965
step =  6000 loss value =  2.771170055381027
step =  7000 loss value =  2.7709113359873823
step =  8000 loss value =  2.7706218302544015
step =  9000 loss value =  2.770295018741801
step =  10000 loss value =  2.7699229959823084
step =  11000 loss value =  2.769496015040127
step =  12000 loss value =  2.7690018562826206
step =  13000 loss value =  2.7684249356823036
step =  14000 loss value =  2.7677450194134323
step =  15000 loss value =  2.766935329868531
step =  16000 loss value =  2.7659596871037433
step =  17000 loss value =  2.7647680782816324
step =  18000 loss value =  2.7632895845312846
step =  19000 loss value =  2.761420710061363
step =  20000 loss value =  2.7590054014357

In [31]:
test_data = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])
for data in test_data:
    sigmoid_val, logical_val = XOR_obj.predict(data)
    print(data, '=', logical_val)

print('----------------------------------------')
test_tdata = np.array([0, 1, 1, 0]) # 정답
accuracy_ret = XOR_obj.accuracy(test_data, test_tdata)
print('Accuracy =>', accuracy_ret)

[0 0] = 0
[0 1] = 0
[1 0] = 1
[1 1] = 0
----------------------------------------
Accuracy => 0.75


In [32]:
# 학습이 더디게 되기 때문에 학습이 덜 된 상태이기에 1만번 돌렸을 때, 제대로 나오지 않는다
# 이런 경우에는 학습의 수를 증가시켜야한다

# 횟수를 늘리니 손실값이 변화하긴하는데 아주 천천히 변한다.
# 은닉층의 개수, 노드의 개수 = 몇개인지 찾는게 최적의 값을 찾는 과정인데 노가다임 ㅠ

# [중요]
#### 즉, 층이 증가하면 학습의 속도는 느려진다.
#### 층이 많아지게 되면 복잡한문제를 더 잘 풀 수 있다.